In [9]:
import pandas as pd
from pathlib import Path

# Load the processed knowledge base
kb_path = Path("data/processed/knowledge_base_chunks.csv")
kb = pd.read_csv(kb_path)

print("Knowledge base loaded successfully!")
print("Number of chunks:", len(kb))
print()
kb.head()

Knowledge base loaded successfully!
Number of chunks: 1756



,Document_ID,Source,Publication_Year,Page_Number,Chunk_ID,Language,Chunk_Text
0,Nigeria_MPCDSR_2022.pdf,"Federal Ministry of Health, Nigeria",2022,2,Nigeria_MPCDSR_2022.pdf_2_1,English,"FOREWORD \nReporting and tracking maternal, pe..."
1,Nigeria_MPCDSR_2022.pdf,"Federal Ministry of Health, Nigeria",2022,2,Nigeria_MPCDSR_2022.pdf_2_2,English,n and strengthening of \nthe health system blo...
2,Nigeria_MPCDSR_2022.pdf,"Federal Ministry of Health, Nigeria",2022,2,Nigeria_MPCDSR_2022.pdf_2_3,English,care providers in providing quality mater...
3,Nigeria_MPCDSR_2022.pdf,"Federal Ministry of Health, Nigeria",2022,2,Nigeria_MPCDSR_2022.pdf_2_4,English,ral Republic of Nigeria
4,Nigeria_MPCDSR_2022.pdf,"Federal Ministry of Health, Nigeria",2022,3,Nigeria_MPCDSR_2022.pdf_3_1,English,ACKNOWLEDGEMENT \nThe Federal Ministry of Heal...


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create the TF-IDF vectorizer
vectorizer = TfidfVectorizer(stop_words="english")

# Convert all chunk texts into searchable vectors
X = vectorizer.fit_transform(kb["Chunk_Text"].fillna(""))

print("TF-IDF vectorization completed successfully!")
print("Matrix shape:", X.shape)

TF-IDF vectorization completed successfully!
Matrix shape: (1756, 9988)


In [7]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_chunks(query, top_k=3):
    """
    Retrieve the most relevant knowledge-base chunks for a user query.
    """
    # Convert the user query into a TF-IDF vector
    query_vec = vectorizer.transform([query])

    # Compute similarity between the query and every knowledge-base chunk
    scores = cosine_similarity(query_vec, X).flatten()

    # Get the indices of the top-ranked chunks
    top_indices = scores.argsort()[-top_k:][::-1]

    # Return the ranked results
    results = kb.iloc[top_indices].copy()
    results["Relevance_Score"] = scores[top_indices]

    return results[["Document_ID", "Page_Number", "Relevance_Score", "Chunk_Text"]]

print("Retrieval function created successfully.")

Retrieval function created successfully.


In [4]:
results = retrieve_chunks("I am bleeding during pregnancy", top_k=3)

results

,Document_ID,Page_Number,Relevance_Score,Chunk_Text
1073,WHO_Essential_Practice_2023.pdf,25,0.501885,VAGINAL BLEEDING\n\n Assess pregnancy status...
1129,WHO_Essential_Practice_2023.pdf,39,0.419068,BLEEDING IN EARLY PREGNANCY AND POST-ABORTION ...
1130,WHO_Essential_Practice_2023.pdf,39,0.400064,Abdominal pain/tenderness \n \n→Temperature >3...


In [5]:
def answer_query(query):
    results = retrieve_chunks(query, top_k=1)
    top = results.iloc[0]

    print("User query:", query)
    print("\nRetrieved evidence:\n")
    print(top["Chunk_Text"][:800])
    print("\nSource:", top["Document_ID"])
    print("Page:", int(top["Page_Number"]))
    print("Relevance score:", round(top["Relevance_Score"], 3))

answer_query("I am bleeding during pregnancy")

User query: I am bleeding during pregnancy

Retrieved evidence:

VAGINAL BLEEDING

 Assess pregnancy status

 Assess amount of bleeding
PREGNANCY STATUS
BLEEDING
TREATMENT
EARLY PREGNANCY
not aware of pregnancy, or not pregnant (uterus 
NOT above umbilicus)
HEAVY BLEEDING
Pad or cloth soaked in < 5 minutes.

 Insert an IV line B9 .

 Give fluids rapidly B9 .

 Give 0.2 mg ergometrine IM B10.

 Repeat 0.2 mg ergometrine IM/IV if bleeding continues.

 If suspect possible complicated abortion, give appropriate IM/IV antibiotics B15.

 Refer woman urgently to hospital B17.
This may be abortion, 
menorrhagia, 
ectopic pregnancy.
LIGHT BLEEDING

 Examine woman as on B19.

 If pregnancy not likely, refer to other clinical guidelines.
LATE PREGNANCY
(uterus above umbilicus)
ANY BLEEDING IS DANGEROUS
DO NOT do vaginal examination, but:

 In

Source: WHO_Essential_Practice_2023.pdf
Page: 25
Relevance score: 0.502


In [6]:
def answer_query(query):
    results = retrieve_chunks(query, top_k=1)
    top = results.iloc[0]

    print("User query:", query)
    print("\nRetrieved evidence:\n")
    print(top["Chunk_Text"][:600])
    print("\nSource:", top["Document_ID"])
    print("Page:", int(top["Page_Number"]))
    print("Relevance score:", round(top["Relevance_Score"], 3))

answer_query("I am bleeding during pregnancy")

User query: I am bleeding during pregnancy

Retrieved evidence:

VAGINAL BLEEDING

 Assess pregnancy status

 Assess amount of bleeding
PREGNANCY STATUS
BLEEDING
TREATMENT
EARLY PREGNANCY
not aware of pregnancy, or not pregnant (uterus 
NOT above umbilicus)
HEAVY BLEEDING
Pad or cloth soaked in < 5 minutes.

 Insert an IV line B9 .

 Give fluids rapidly B9 .

 Give 0.2 mg ergometrine IM B10.

 Repeat 0.2 mg ergometrine IM/IV if bleeding continues.

 If suspect possible complicated abortion, give appropriate IM/IV antibiotics B15.

 Refer woman urgently to hospital B17.
This may be abortion, 
menorrhagia, 
ectopic pregnancy.
LIGHT BLEEDING



Source: WHO_Essential_Practice_2023.pdf
Page: 25
Relevance score: 0.502


In [63]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_by_category(query, category, top_k=3):
    """
    Retrieve the most relevant knowledge-base chunks for a detected danger-sign category.
    Returns the top-ranked chunks with document, page number, relevance score, and text.
    """

    # -----------------------------
    # Filter the knowledge base by danger-sign category
    # -----------------------------
    if category == "Vaginal bleeding":
        filtered = kb[
            (kb["Document_ID"].str.contains("WHO_Essential_Practice_2023", case=False, na=False))
            &
            (kb["Chunk_Text"].str.contains(
                r"vaginal bleeding|bleeding|blood|haemorrhage|hemorrhage|ANY BLEEDING IS DANGEROUS",
                case=False,
                na=False,
                regex=True
            ))
        ]

    elif category == "Severe headache / possible pre-eclampsia":
        filtered = kb[
            (kb["Document_ID"].str.contains("WHO_Essential_Practice_2023", case=False, na=False))
            &
            (kb["Chunk_Text"].str.contains(
                r"pre-eclampsia|eclampsia|CHECK FOR PRE-ECLAMPSIA|blood pressure|blurred vision|vision|convulsion|protein in urine|pregnant woman|pregnancy status",
                case=False,
                na=False,
                regex=True
            ))
        ]

    elif category == "Reduced fetal movement":
        filtered = kb[
            (kb["Document_ID"].str.contains("WHO_Antenatal_Care_2016", case=False, na=False))
            &
            (kb["Chunk_Text"].str.contains(
                r"fetal movement|foetal movement|baby move|movement|kick|stopped moving|decreased movement",
                case=False,
                na=False,
                regex=True
            ))
        ]

    else:
        filtered = kb

    # -----------------------------
    # Safety fallback if filtering returns no rows
    # -----------------------------
    if filtered.empty:
        filtered = kb

    # -----------------------------
    # Build TF-IDF model on the filtered subset
    # -----------------------------
    vectorizer_local = TfidfVectorizer(stop_words="english")
    X_local = vectorizer_local.fit_transform(filtered["Chunk_Text"].fillna(""))

    query_vector = vectorizer_local.transform([query])

    # Convert similarity scores to a NumPy array
    scores = np.array(cosine_similarity(query_vector, X_local).flatten())

    # -----------------------------
    # Clinical boosting
    # -----------------------------
    if category == "Severe headache / possible pre-eclampsia":
        boost = filtered["Chunk_Text"].str.contains(
            r"CHECK FOR PRE-ECLAMPSIA|pre-eclampsia|eclampsia",
            case=False,
            na=False,
            regex=True
        ).astype(float).to_numpy() * 0.5
        scores = scores + boost

    elif category == "Vaginal bleeding":
        boost = filtered["Chunk_Text"].str.contains(
            r"ANY BLEEDING IS DANGEROUS|vaginal bleeding|bleeding",
            case=False,
            na=False,
            regex=True
        ).astype(float).to_numpy() * 0.5
        scores = scores + boost

    # -----------------------------
    # Select top-ranked chunks
    # -----------------------------
    top_indices = scores.argsort()[-top_k:][::-1]

    results = filtered.iloc[top_indices].copy()
    results["Relevance_Score"] = scores[top_indices]

    return results[["Document_ID", "Page_Number", "Relevance_Score", "Chunk_Text"]]

In [64]:
result = mamacare_rag_assistant("My head is hurting badly and my vision is blurry.")

for k, v in result.items():
    print(f"{k}: {v}\n")

language: english

category: Severe headache / possible pre-eclampsia

response: Based on the retrieved WHO guidance, this may be a serious pregnancy danger sign. Please go to the nearest hospital or health facility immediately.

evidence: CHECK FOR PRE-ECLAMPSIA Screen all pregnant women at every visit. ASK, CHECK RECORD LOOK, LISTEN, FEEL SIGNS CLASSIFY TREAT AND ADVISE Assess gestational age Blood pressure at the last visit? Eclampsia or pre-eclampsia in previous pregnancies? Multiple pregnancies? Other diseases (chronic hypertension, kidney disease or autoimmune disease)? Measure blood pressure in sitting position (her legs should not be dangling or crossed. Her feet should be supported or on the ground. The elbow (brachial artery) should be at the level of the heart. Ensure that the cuff is neither too wide nor too narrow).

source: WHO_Essential_Practice_2023.pdf

page: 46

score: 0.596

escalation: Level 3 - Urgent / Emergency referral



In [65]:
result = mamacare_rag_assistant("I have a severe headache during pregnancy")

for k, v in result.items():
    print(f"{k}: {v}\n")

language: english

category: Severe headache / possible pre-eclampsia

response: Based on the retrieved WHO guidance, this may be a serious pregnancy danger sign. Please go to the nearest hospital or health facility immediately.

evidence: Feel for transverse lie. Listen to fetal heart. t Next: Check for pre-eclampsia ANTENATAL CARE C2 Assess the pregnant woman f Pregnancy status, birth and emergency plan C2 ASSESS THE PREGNANT WOMAN: PREGNANCY STATUS, BIRTH AND EMERGENCY PLAN CHECK FOR PRE-ECLAMPSIA Screen all pregnant women at every visit. ASK, CHECK RECORD LOOK, LISTEN, FEEL SIGNS CLASSIFY TREAT AND ADVISE Blood pressure at the last visit? Eclampsia or pre-eclampsia in previous pregnancies? Multiple pregnancies? Other diseases (chronic hypertension, kidney disease or autoimmune disease)? Measure blood pressure in sitting pos

source: WHO_Essential_Practice_2023.pdf

page: 44

score: 0.721

escalation: Level 3 - Urgent / Emergency referral



In [66]:
def detect_language(text):
    text = text.lower()

    pidgin_words = [
        "dey", "abeg", "waka", "body", "well-well",
        "fit", "hospital", "blood dey", "make", "you"
    ]

    score = sum(word in text for word in pidgin_words)

    return "pidgin" if score >= 2 else "english"

print("Language detection function loaded.")

Language detection function loaded.


In [67]:
def mamacare_rag_assistant(query):
    # Step 1: Detect language
    language = detect_language(query)

    # Step 2: Detect danger-sign category
    text = query.lower()

    if "bleeding" in text or "blood" in text:
        category = "Vaginal bleeding"
        escalation = "Level 3 - Urgent / Emergency referral"

    elif (
        "headache" in text or
        "head hurts" in text or
        "head is hurting" in text or
        "hurting badly" in text or
        "head dey pain" in text or
        "strong headache" in text or
        "vision is blurry" in text or
        "blurred vision" in text or
        "my eye dey blur" in text or
        "eye dey blur" in text or
        "body dey swell" in text or
        "hands are swollen" in text or
        "face is swollen" in text or
        "swelling" in text
    ):
        category = "Severe headache / possible pre-eclampsia"
        escalation = "Level 3 - Urgent / Emergency referral"

    elif (
        "baby" in text or
        "fetal movement" in text or
        "movement" in text or
        "no dey move" in text or
        "stopped moving" in text
    ):
        category = "Reduced fetal movement"
        escalation = "Level 3 - Urgent / Emergency referral"

    else:
        category = "Unknown / unsupported"
        escalation = "Level 1 - Safe fallback"

    # Step 3: Retrieve evidence using category-aware retrieval
    results = retrieve_by_category(query, category, top_k=1)
    top = results.iloc[0]

    # Step 4: Generate bilingual response
    if language == "pidgin":
        response = (
            "Based on the WHO information wey the system retrieve, this fit be "
            "serious pregnancy danger sign. Abeg go the nearest hospital or health "
            "centre immediately."
        )
    else:
        response = (
            "Based on the retrieved WHO guidance, this may be a serious pregnancy "
            "danger sign. Please go to the nearest hospital or health facility immediately."
        )

    # Step 5: Clean the retrieved evidence text
    clean_evidence = (
        top["Chunk_Text"]
        .replace("\n", " ")
        .replace("", "")
        .replace("", "")
    )
    clean_evidence = " ".join(clean_evidence.split())[:600]

    # Step 6: Return the result
    return {
        "language": language,
        "category": category,
        "response": response,
        "evidence": clean_evidence,
        "source": top["Document_ID"],
        "page": int(top["Page_Number"]),
        "score": round(top["Relevance_Score"], 3),
        "escalation": escalation,
    }

In [68]:
result = mamacare_rag_assistant("My head is hurting badly and my vision is blurry.")

for k, v in result.items():
    print(f"{k}: {v}\n")

language: english

category: Severe headache / possible pre-eclampsia

response: Based on the retrieved WHO guidance, this may be a serious pregnancy danger sign. Please go to the nearest hospital or health facility immediately.

evidence: CHECK FOR PRE-ECLAMPSIA Screen all pregnant women at every visit. ASK, CHECK RECORD LOOK, LISTEN, FEEL SIGNS CLASSIFY TREAT AND ADVISE Assess gestational age Blood pressure at the last visit? Eclampsia or pre-eclampsia in previous pregnancies? Multiple pregnancies? Other diseases (chronic hypertension, kidney disease or autoimmune disease)? Measure blood pressure in sitting position (her legs should not be dangling or crossed. Her feet should be supported or on the ground. The elbow (brachial artery) should be at the level of the heart. Ensure that the cuff is neither too wide nor too narrow).

source: WHO_Essential_Practice_2023.pdf

page: 46

score: 0.596

escalation: Level 3 - Urgent / Emergency referral



In [69]:
result = mamacare_rag_assistant("I have a severe headache during pregnancy")

for k, v in result.items():
    print(f"{k}: {v}\n")

language: english

category: Severe headache / possible pre-eclampsia

response: Based on the retrieved WHO guidance, this may be a serious pregnancy danger sign. Please go to the nearest hospital or health facility immediately.

evidence: Feel for transverse lie. Listen to fetal heart. t Next: Check for pre-eclampsia ANTENATAL CARE C2 Assess the pregnant woman f Pregnancy status, birth and emergency plan C2 ASSESS THE PREGNANT WOMAN: PREGNANCY STATUS, BIRTH AND EMERGENCY PLAN CHECK FOR PRE-ECLAMPSIA Screen all pregnant women at every visit. ASK, CHECK RECORD LOOK, LISTEN, FEEL SIGNS CLASSIFY TREAT AND ADVISE Blood pressure at the last visit? Eclampsia or pre-eclampsia in previous pregnancies? Multiple pregnancies? Other diseases (chronic hypertension, kidney disease or autoimmune disease)? Measure blood pressure in sitting pos

source: WHO_Essential_Practice_2023.pdf

page: 44

score: 0.721

escalation: Level 3 - Urgent / Emergency referral



In [70]:
result = mamacare_rag_assistant("I am bleeding during pregnancy")

for k, v in result.items():
    print(f"{k}: {v}\n")

language: english

category: Vaginal bleeding

response: Based on the retrieved WHO guidance, this may be a serious pregnancy danger sign. Please go to the nearest hospital or health facility immediately.

evidence: VAGINAL BLEEDING Assess pregnancy status Assess amount of bleeding PREGNANCY STATUS BLEEDING TREATMENT EARLY PREGNANCY not aware of pregnancy, or not pregnant (uterus NOT above umbilicus) HEAVY BLEEDING Pad or cloth soaked in < 5 minutes. Insert an IV line B9 . Give fluids rapidly B9 . Give 0.2 mg ergometrine IM B10. Repeat 0.2 mg ergometrine IM/IV if bleeding continues. If suspect possible complicated abortion, give appropriate IM/IV antibiotics B15. Refer woman urgently to hospital B17. This may be abortion, menorrhagia, ectopic pregnancy. LIGHT BLEEDING Examine woman as on B19. If pregnancy

source: WHO_Essential_Practice_2023.pdf

page: 25

score: 0.917

escalation: Level 3 - Urgent / Emergency referral



In [49]:
import pandas as pd

evaluation_data = [
    # Vaginal bleeding (English)
    {"No":1,"User_Expression":"I am bleeding during pregnancy.","Language":"English","Expression_Type":"Direct","Expected_Category":"Vaginal bleeding","Expected_Escalation":"Level 3 - Urgent / Emergency referral","Supporting_Source":"WHO Essential Practice (2023), p.25"},
    {"No":2,"User_Expression":"There is blood coming from my vagina.","Language":"English","Expression_Type":"Direct","Expected_Category":"Vaginal bleeding","Expected_Escalation":"Level 3 - Urgent / Emergency referral","Supporting_Source":"WHO Essential Practice (2023), p.25"},
    {"No":3,"User_Expression":"I noticed blood on my underwear today.","Language":"English","Expression_Type":"Indirect","Expected_Category":"Vaginal bleeding","Expected_Escalation":"Level 3 - Urgent / Emergency referral","Supporting_Source":"WHO Essential Practice (2023), p.25"},

    # Severe headache (English)
    {"No":4,"User_Expression":"I have a severe headache during pregnancy.","Language":"English","Expression_Type":"Direct","Expected_Category":"Severe headache / possible pre-eclampsia","Expected_Escalation":"Level 3 - Urgent / Emergency referral","Supporting_Source":"WHO Essential Practice (2023), p.44"},
    {"No":5,"User_Expression":"My head is hurting badly and my vision is blurry.","Language":"English","Expression_Type":"Indirect","Expected_Category":"Severe headache / possible pre-eclampsia","Expected_Escalation":"Level 3 - Urgent / Emergency referral","Supporting_Source":"WHO Essential Practice (2023), p.44"},

    # Reduced fetal movement (English)
    {"No":6,"User_Expression":"My baby has stopped moving today.","Language":"English","Expression_Type":"Direct","Expected_Category":"Reduced fetal movement","Expected_Escalation":"Level 3 - Urgent / Emergency referral","Supporting_Source":"WHO Antenatal Care (2016)"},
    {"No":7,"User_Expression":"I have not felt the baby move for several hours.","Language":"English","Expression_Type":"Indirect","Expected_Category":"Reduced fetal movement","Expected_Escalation":"Level 3 - Urgent / Emergency referral","Supporting_Source":"WHO Antenatal Care (2016)"},

    # Vaginal bleeding (Pidgin)
    {"No":8,"User_Expression":"Blood dey come out during pregnancy.","Language":"Nigerian Pidgin","Expression_Type":"Direct","Expected_Category":"Vaginal bleeding","Expected_Escalation":"Level 3 - Urgent / Emergency referral","Supporting_Source":"WHO Essential Practice (2023), p.25"},
    {"No":9,"User_Expression":"I see blood for my wrapper today.","Language":"Nigerian Pidgin","Expression_Type":"Indirect","Expected_Category":"Vaginal bleeding","Expected_Escalation":"Level 3 - Urgent / Emergency referral","Supporting_Source":"WHO Essential Practice (2023), p.25"},

    # Severe headache (Pidgin)
    {"No":10,"User_Expression":"Head dey pain me well-well.","Language":"Nigerian Pidgin","Expression_Type":"Direct","Expected_Category":"Severe headache / possible pre-eclampsia","Expected_Escalation":"Level 3 - Urgent / Emergency referral","Supporting_Source":"WHO Essential Practice (2023), p.44"},
    {"No":11,"User_Expression":"My eye dey blur and my body dey swell.","Language":"Nigerian Pidgin","Expression_Type":"Indirect","Expected_Category":"Severe headache / possible pre-eclampsia","Expected_Escalation":"Level 3 - Urgent / Emergency referral","Supporting_Source":"WHO Essential Practice (2023), p.44"},

    # Reduced fetal movement (Pidgin)
    {"No":12,"User_Expression":"My baby no dey move today.","Language":"Nigerian Pidgin","Expression_Type":"Direct","Expected_Category":"Reduced fetal movement","Expected_Escalation":"Level 3 - Urgent / Emergency referral","Supporting_Source":"WHO Antenatal Care (2016)"},

    # Ambiguous / unsupported
    {"No":13,"User_Expression":"I feel tired today.","Language":"English","Expression_Type":"Ambiguous","Expected_Category":"Unknown / unsupported","Expected_Escalation":"Level 1 - Safe fallback","Supporting_Source":"No direct WHO danger-sign evidence"},
    {"No":14,"User_Expression":"I no just feel fine.","Language":"Nigerian Pidgin","Expression_Type":"Ambiguous","Expected_Category":"Unknown / unsupported","Expected_Escalation":"Level 1 - Safe fallback","Supporting_Source":"No direct WHO danger-sign evidence"}
]

eval_df = pd.DataFrame(evaluation_data)
eval_df

,No,User_Expression,Language,Expression_Type,Expected_Category,Expected_Escalation,Supporting_Source
0,1,I am bleeding during pregnancy.,English,Direct,Vaginal bleeding,Level 3 - Urgent / Emergency referral,"WHO Essential Practice (2023), p.25"
1,2,There is blood coming from my vagina.,English,Direct,Vaginal bleeding,Level 3 - Urgent / Emergency referral,"WHO Essential Practice (2023), p.25"
2,3,I noticed blood on my underwear today.,English,Indirect,Vaginal bleeding,Level 3 - Urgent / Emergency referral,"WHO Essential Practice (2023), p.25"
3,4,I have a severe headache during pregnancy.,English,Direct,Severe headache / possible pre-eclampsia,Level 3 - Urgent / Emergency referral,"WHO Essential Practice (2023), p.44"
4,5,My head is hurting badly and my vision is blurry.,English,Indirect,Severe headache / possible pre-eclampsia,Level 3 - Urgent / Emergency referral,"WHO Essential Practice (2023), p.44"
5,6,My baby has stopped moving today.,English,Direct,Reduced fetal movement,Level 3 - Urgent / Emergency referral,WHO Antenatal Care (2016)
6,7,I have not felt the baby move for several hours.,English,Indirect,Reduced fetal movement,Level 3 - Urgent / Emergency referral,WHO Antenatal Care (2016)
7,8,Blood dey come out during pregnancy.,Nigerian Pidgin,Direct,Vaginal bleeding,Level 3 - Urgent / Emergency referral,"WHO Essential Practice (2023), p.25"
8,9,I see blood for my wrapper today.,Nigerian Pidgin,Indirect,Vaginal bleeding,Level 3 - Urgent / Emergency referral,"WHO Essential Practice (2023), p.25"
9,10,Head dey pain me well-well.,Nigerian Pidgin,Direct,Severe headache / possible pre-eclampsia,Level 3 - Urgent / Emergency referral,"WHO Essential Practice (2023), p.44"


In [71]:
import os

os.makedirs("data/evaluation", exist_ok=True)

eval_df.to_csv("data/evaluation/bilingual_evaluation_dataset.csv", index=False)

print("Evaluation dataset saved successfully!")
print("Location: data/evaluation/bilingual_evaluation_dataset.csv")

Evaluation dataset saved successfully!
Location: data/evaluation/bilingual_evaluation_dataset.csv


In [72]:
results = []

for _, row in eval_df.iterrows():
    output = mamacare_rag_assistant(row["User_Expression"])

    results.append({
        "No": row["No"],
        "Query": row["User_Expression"],
        "Expected_Category": row["Expected_Category"],
        "Predicted_Category": output["category"],
        "Expected_Escalation": row["Expected_Escalation"],
        "Predicted_Escalation": output["escalation"],
        "Retrieved_Source": output["source"],
        "Retrieved_Page": output["page"]
    })

results_df = pd.DataFrame(results)
results_df

,No,Query,Expected_Category,Predicted_Category,Expected_Escalation,Predicted_Escalation,Retrieved_Source,Retrieved_Page
0,1,I am bleeding during pregnancy.,Vaginal bleeding,Vaginal bleeding,Level 3 - Urgent / Emergency referral,Level 3 - Urgent / Emergency referral,WHO_Essential_Practice_2023.pdf,25
1,2,There is blood coming from my vagina.,Vaginal bleeding,Vaginal bleeding,Level 3 - Urgent / Emergency referral,Level 3 - Urgent / Emergency referral,WHO_Essential_Practice_2023.pdf,167
2,3,I noticed blood on my underwear today.,Vaginal bleeding,Vaginal bleeding,Level 3 - Urgent / Emergency referral,Level 3 - Urgent / Emergency referral,WHO_Essential_Practice_2023.pdf,64
3,4,I have a severe headache during pregnancy.,Severe headache / possible pre-eclampsia,Severe headache / possible pre-eclampsia,Level 3 - Urgent / Emergency referral,Level 3 - Urgent / Emergency referral,WHO_Essential_Practice_2023.pdf,44
4,5,My head is hurting badly and my vision is blurry.,Severe headache / possible pre-eclampsia,Severe headache / possible pre-eclampsia,Level 3 - Urgent / Emergency referral,Level 3 - Urgent / Emergency referral,WHO_Essential_Practice_2023.pdf,46
5,6,My baby has stopped moving today.,Reduced fetal movement,Reduced fetal movement,Level 3 - Urgent / Emergency referral,Level 3 - Urgent / Emergency referral,WHO_Antenatal_Care_2016.pdf,71
6,7,I have not felt the baby move for several hours.,Reduced fetal movement,Reduced fetal movement,Level 3 - Urgent / Emergency referral,Level 3 - Urgent / Emergency referral,WHO_Antenatal_Care_2016.pdf,72
7,8,Blood dey come out during pregnancy.,Vaginal bleeding,Vaginal bleeding,Level 3 - Urgent / Emergency referral,Level 3 - Urgent / Emergency referral,WHO_Essential_Practice_2023.pdf,25
8,9,I see blood for my wrapper today.,Vaginal bleeding,Vaginal bleeding,Level 3 - Urgent / Emergency referral,Level 3 - Urgent / Emergency referral,WHO_Essential_Practice_2023.pdf,64
9,10,Head dey pain me well-well.,Severe headache / possible pre-eclampsia,Severe headache / possible pre-eclampsia,Level 3 - Urgent / Emergency referral,Level 3 - Urgent / Emergency referral,WHO_Essential_Practice_2023.pdf,46


In [74]:
category_accuracy = (
    results_df["Expected_Category"] ==
    results_df["Predicted_Category"]
).mean()

escalation_accuracy = (
    results_df["Expected_Escalation"] ==
    results_df["Predicted_Escalation"]
).mean()

print(f"Category accuracy: {category_accuracy:.2%}")
print(f"Escalation accuracy: {escalation_accuracy:.2%}")

Category accuracy: 100.00%
Escalation accuracy: 100.00%


In [76]:
import os

os.makedirs("data/results", exist_ok=True)

results_df.to_csv("data/results/evaluation_results.csv", index=False)

print("Evaluation results saved successfully!")
print("Location: data/results/evaluation_results.csv")

Evaluation results saved successfully!
Location: data/results/evaluation_results.csv


## Evaluation Summary

The bilingual retrieval-and-safety pipeline was evaluated using a labelled dataset containing **14 test cases** in **Simple English** and **Nigerian Pidgin**. The dataset included direct, indirect, and ambiguous expressions across the three implemented priority danger signs.

### Results

* **Danger-sign category accuracy:** 100.00%
* **Safety escalation accuracy:** 100.00%

The system correctly detected all danger-sign categories, retrieved supporting WHO evidence, returned the corresponding source document and page number, and applied the appropriate safety escalation decision for every evaluation case.


In [78]:
# Add the language column from the evaluation dataset
results_df["Language"] = eval_df["Language"]

results_df.head()

,No,Query,Expected_Category,Predicted_Category,Expected_Escalation,Predicted_Escalation,Retrieved_Source,Retrieved_Page,Language
0,1,I am bleeding during pregnancy.,Vaginal bleeding,Vaginal bleeding,Level 3 - Urgent / Emergency referral,Level 3 - Urgent / Emergency referral,WHO_Essential_Practice_2023.pdf,25,English
1,2,There is blood coming from my vagina.,Vaginal bleeding,Vaginal bleeding,Level 3 - Urgent / Emergency referral,Level 3 - Urgent / Emergency referral,WHO_Essential_Practice_2023.pdf,167,English
2,3,I noticed blood on my underwear today.,Vaginal bleeding,Vaginal bleeding,Level 3 - Urgent / Emergency referral,Level 3 - Urgent / Emergency referral,WHO_Essential_Practice_2023.pdf,64,English
3,4,I have a severe headache during pregnancy.,Severe headache / possible pre-eclampsia,Severe headache / possible pre-eclampsia,Level 3 - Urgent / Emergency referral,Level 3 - Urgent / Emergency referral,WHO_Essential_Practice_2023.pdf,44,English
4,5,My head is hurting badly and my vision is blurry.,Severe headache / possible pre-eclampsia,Severe headache / possible pre-eclampsia,Level 3 - Urgent / Emergency referral,Level 3 - Urgent / Emergency referral,WHO_Essential_Practice_2023.pdf,46,English


In [79]:
# Identify danger-sign and non-danger-sign cases
danger_cases = results_df[results_df["Expected_Category"] != "Unknown / unsupported"]
non_danger_cases = results_df[results_df["Expected_Category"] == "Unknown / unsupported"]

# True positives: danger sign correctly detected
TP = (danger_cases["Expected_Category"] == danger_cases["Predicted_Category"]).sum()

# False negatives: danger sign missed
FN = (danger_cases["Expected_Category"] != danger_cases["Predicted_Category"]).sum()

# False positives: non-danger sign incorrectly classified as a danger sign
FP = (non_danger_cases["Predicted_Category"] != "Unknown / unsupported").sum()

# True negatives: non-danger sign correctly left unsupported
TN = (non_danger_cases["Predicted_Category"] == "Unknown / unsupported").sum()

recall = TP / (TP + FN)
false_negative_rate = FN / (TP + FN)
false_positive_rate = FP / (FP + TN)

print(f"True Positives (TP): {TP}")
print(f"False Negatives (FN): {FN}")
print(f"False Positives (FP): {FP}")
print(f"True Negatives (TN): {TN}")
print()
print(f"Danger-sign Recall: {recall:.2%}")
print(f"False-Negative Rate: {false_negative_rate:.2%}")
print(f"False-Positive Rate: {false_positive_rate:.2%}")

True Positives (TP): 12
False Negatives (FN): 0
False Positives (FP): 0
True Negatives (TN): 2

Danger-sign Recall: 100.00%
False-Negative Rate: 0.00%
False-Positive Rate: 0.00%


In [80]:
# Per-category recall
category_results = []

for category in danger_cases["Expected_Category"].unique():
    subset = danger_cases[danger_cases["Expected_Category"] == category]
    tp = (subset["Expected_Category"] == subset["Predicted_Category"]).sum()
    fn = (subset["Expected_Category"] != subset["Predicted_Category"]).sum()
    recall = tp / (tp + fn)

    category_results.append({
        "Category": category,
        "Cases": len(subset),
        "Recall": recall
    })

category_df = pd.DataFrame(category_results)
category_df

,Category,Cases,Recall
0,Vaginal bleeding,5,1.0
1,Severe headache / possible pre-eclampsia,4,1.0
2,Reduced fetal movement,3,1.0


In [81]:
# Performance by language
language_results = []

for lang in results_df["Language"].unique():
    subset = results_df[results_df["Language"] == lang]
    accuracy = (subset["Expected_Category"] == subset["Predicted_Category"]).mean()

    language_results.append({
        "Language": lang,
        "Cases": len(subset),
        "Accuracy": accuracy
    })

language_df = pd.DataFrame(language_results)
language_df

,Language,Cases,Accuracy
0,English,8,1.0
1,Nigerian Pidgin,6,1.0
